# 07_a Exercises: Regression in Time Series (Solutions)

Worked solution for ARIMA and Prophet forecasting with the sunspots monthly dataset.

In [ ]:
import pandas as pd

df_sunspots = pd.read_csv('./data/sunspots_TS/sunspots.csv', parse_dates=['Date']).drop(columns=['Unnamed: 0'], axis=1)
df_sunspots = df_sunspots[['Date', 'Monthly Mean Total Sunspot Number']].copy()

display(df_sunspots.head())
df_sunspots.info()


In [ ]:
horizon = 24
series = df_sunspots.set_index('Date')['Monthly Mean Total Sunspot Number']

train = series.iloc[:-horizon]
test = series.iloc[-horizon:]

print(f'Train size: {len(train)}')
print(f'Test size: {len(test)}')


In [ ]:
import statsmodels.api as sm

arima_baseline = sm.tsa.arima.ARIMA(series, order=(1, 1, 1)).fit()
print(arima_baseline.summary())


In [ ]:
import numpy as np

p_range = range(0, 4)
d_range = range(0, 3)
q_range = range(0, 4)

best_order = None
best_model = None
best_aic = float('inf')

for p in p_range:
    for d in d_range:
        for q in q_range:
            try:
                model = sm.tsa.arima.ARIMA(train, order=(p, d, q)).fit()
                if model.aic < best_aic:
                    best_aic = model.aic
                    best_order = (p, d, q)
                    best_model = model
            except Exception:
                continue

print(f'Best ARIMA order: {best_order}')
print(f'Best train AIC: {best_aic:.2f}')


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

arima_forecast = best_model.forecast(steps=len(test))

arima_mae = mean_absolute_error(test, arima_forecast)
arima_rmse = np.sqrt(mean_squared_error(test, arima_forecast))
arima_mape = np.mean(np.abs((test - arima_forecast) / test.replace(0, np.nan))) * 100

print(f'ARIMA MAE:  {arima_mae:.2f}')
print(f'ARIMA RMSE: {arima_rmse:.2f}')
print(f'ARIMA MAPE: {arima_mape:.2f}%')


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(test.index, test.values, label='Real (Test)', marker='o')
plt.plot(test.index, arima_forecast.values, label='Predicted (ARIMA)', linestyle='--', marker='o')
plt.title(f'ARIMA: Real vs Predicted (order={best_order})')
plt.xlabel('Date')
plt.ylabel('Sunspot Number')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from prophet import Prophet

prophet_df = df_sunspots.rename(columns={'Date': 'ds', 'Monthly Mean Total Sunspot Number': 'y'})
train_df = prophet_df.iloc[:-horizon].copy()
test_df = prophet_df.iloc[-horizon:].copy()

prophet_model = Prophet()
prophet_model.fit(train_df)

future = prophet_model.make_future_dataframe(periods=horizon, freq='M')
prophet_forecast_full = prophet_model.predict(future)


In [ ]:
pred_test = prophet_forecast_full[['ds', 'yhat']].tail(horizon).copy()
comparison = test_df[['ds', 'y']].merge(pred_test, on='ds', how='left')

prophet_mae = mean_absolute_error(comparison['y'], comparison['yhat'])
prophet_rmse = np.sqrt(mean_squared_error(comparison['y'], comparison['yhat']))
prophet_mape = np.mean(np.abs((comparison['y'] - comparison['yhat']) / comparison['y'].replace(0, np.nan))) * 100

print(f'Prophet MAE:  {prophet_mae:.2f}')
print(f'Prophet RMSE: {prophet_rmse:.2f}')
print(f'Prophet MAPE: {prophet_mape:.2f}%')

plt.figure(figsize=(12, 5))
plt.plot(comparison['ds'], comparison['y'], label='Real', marker='o')
plt.plot(comparison['ds'], comparison['yhat'], label='Predicted (Prophet)', linestyle='--', marker='o')
plt.title('Prophet: Real vs Predicted (Test Horizon)')
plt.xlabel('Date')
plt.ylabel('Sunspot Number')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
print('Model comparison on test set:')
print(f'ARIMA   -> MAE: {arima_mae:.2f}, RMSE: {arima_rmse:.2f}, MAPE: {arima_mape:.2f}%')
print(f'Prophet -> MAE: {prophet_mae:.2f}, RMSE: {prophet_rmse:.2f}, MAPE: {prophet_mape:.2f}%')


In [ ]:
# Build lag features for supervised ML models
lags = 12
lagged_df = pd.DataFrame({'y': series})
for lag in range(1, lags + 1):
    lagged_df[f'lag_{lag}'] = lagged_df['y'].shift(lag)

lagged_df = lagged_df.dropna()

X_ml = lagged_df.drop(columns=['y'])
y_ml = lagged_df['y']

horizon_ml = horizon
X_train_ml, X_test_ml = X_ml.iloc[:-horizon_ml], X_ml.iloc[-horizon_ml:]
y_train_ml, y_test_ml = y_ml.iloc[:-horizon_ml], y_ml.iloc[-horizon_ml:]


In [ ]:
# Neural Network (MLPRegressor)
from sklearn.neural_network import MLPRegressor

nn_model = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=42)
nn_model.fit(X_train_ml, y_train_ml)
nn_pred = pd.Series(nn_model.predict(X_test_ml), index=y_test_ml.index)

nn_mae = mean_absolute_error(y_test_ml, nn_pred)
nn_rmse = np.sqrt(mean_squared_error(y_test_ml, nn_pred))
nn_mape = np.mean(np.abs((y_test_ml - nn_pred) / y_test_ml.replace(0, np.nan))) * 100

print(f'NN MAE:  {nn_mae:.2f}')
print(f'NN RMSE: {nn_rmse:.2f}')
print(f'NN MAPE: {nn_mape:.2f}%')

plt.figure(figsize=(12, 5))
plt.plot(y_test_ml.index, y_test_ml.values, label='Real', marker='o')
plt.plot(nn_pred.index, nn_pred.values, label='Predicted (NN)', linestyle='--', marker='o')
plt.title('NN: Real vs Predicted')
plt.xlabel('Date')
plt.ylabel('Sunspot Number')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
rf_model.fit(X_train_ml, y_train_ml)
rf_pred = pd.Series(rf_model.predict(X_test_ml), index=y_test_ml.index)

rf_mae = mean_absolute_error(y_test_ml, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test_ml, rf_pred))
rf_mape = np.mean(np.abs((y_test_ml - rf_pred) / y_test_ml.replace(0, np.nan))) * 100

print(f'RF MAE:  {rf_mae:.2f}')
print(f'RF RMSE: {rf_rmse:.2f}')
print(f'RF MAPE: {rf_mape:.2f}%')

plt.figure(figsize=(12, 5))
plt.plot(y_test_ml.index, y_test_ml.values, label='Real', marker='o')
plt.plot(rf_pred.index, rf_pred.values, label='Predicted (RF)', linestyle='--', marker='o')
plt.title('RF: Real vs Predicted')
plt.xlabel('Date')
plt.ylabel('Sunspot Number')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# LSTM forecasting
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense

tf.random.set_seed(42)

window = 12
values = series.values.astype('float32')

X_seq, y_seq = [], []
for i in range(window, len(values)):
    X_seq.append(values[i-window:i])
    y_seq.append(values[i])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)
seq_index = series.index[window:]

split_idx = len(X_seq) - horizon
X_train_seq, X_test_seq = X_seq[:split_idx], X_seq[split_idx:]
y_train_seq, y_test_seq = y_seq[:split_idx], y_seq[split_idx:]
test_seq_index = seq_index[split_idx:]

X_train_seq = X_train_seq.reshape((X_train_seq.shape[0], X_train_seq.shape[1], 1))
X_test_seq = X_test_seq.reshape((X_test_seq.shape[0], X_test_seq.shape[1], 1))

lstm_model = Sequential([
    LSTM(32, input_shape=(window, 1)),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

lstm_model.fit(X_train_seq, y_train_seq, epochs=50, batch_size=16, verbose=0)
lstm_pred = lstm_model.predict(X_test_seq, verbose=0).flatten()
lstm_pred = pd.Series(lstm_pred, index=test_seq_index)
y_test_seq_series = pd.Series(y_test_seq, index=test_seq_index)

lstm_mae = mean_absolute_error(y_test_seq_series, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test_seq_series, lstm_pred))
lstm_mape = np.mean(np.abs((y_test_seq_series - lstm_pred) / y_test_seq_series.replace(0, np.nan))) * 100

print(f'LSTM MAE:  {lstm_mae:.2f}')
print(f'LSTM RMSE: {lstm_rmse:.2f}')
print(f'LSTM MAPE: {lstm_mape:.2f}%')

plt.figure(figsize=(12, 5))
plt.plot(y_test_seq_series.index, y_test_seq_series.values, label='Real', marker='o')
plt.plot(lstm_pred.index, lstm_pred.values, label='Predicted (LSTM)', linestyle='--', marker='o')
plt.title('LSTM: Real vs Predicted')
plt.xlabel('Date')
plt.ylabel('Sunspot Number')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'NN', 'RF', 'LSTM'],
    'MAE': [arima_mae, prophet_mae, nn_mae, rf_mae, lstm_mae],
    'RMSE': [arima_rmse, prophet_rmse, nn_rmse, rf_rmse, lstm_rmse],
    'MAPE': [arima_mape, prophet_mape, nn_mape, rf_mape, lstm_mape]
}).sort_values('RMSE')

display(comparison_df)
